# Clase 19 — Lab: EXPLAIN ANALYZE

**Diseño de Índices en la Práctica** · IIC 2413 — Bases de Datos 

## Setup

Requisitos:

- **Postgres.app** corriendo. Si usas postgresql normal, adapta el código. 
- psycopg2

Vamos a crear una base nueva `lab_explain` y conectarnos con `psycopg2`.

In [ ]:
import os, subprocess

# Asegura que el psql de Postgres.app esté disponible para los !comandos.
PG_BIN = "/Applications/Postgres.app/Contents/Versions/latest/bin"
if PG_BIN not in os.environ.get("PATH", ""):
    os.environ["PATH"] = PG_BIN + os.pathsep + os.environ.get("PATH", "")

DB = "lab_explain"
USER = os.environ.get("USER", "postgres")

# Recrea la base limpia.
subprocess.run(["dropdb", "--if-exists", DB], check=True)
subprocess.run(["createdb", DB], check=True)
print(f"Base de datos {DB} creada.")

In [ ]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(dbname=DB, host="localhost", user=USER)
conn.autocommit = True

def run(sql):
    """Ejecuta una o más sentencias SQL. Si la última devuelve filas, las
    retorna como DataFrame; si no, no retorna nada."""
    with conn.cursor() as cur:
        cur.execute(sql)
        if cur.description is not None:
            cols = [d.name for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)

def explain(sql):
    """Imprime el plan de un EXPLAIN/EXPLAIN ANALYZE línea por línea."""
    df = run(sql)
    print("\n".join(df.iloc[:, 0].tolist()))

## Parte 1 · EXPLAIN vs. EXPLAIN ANALYZE

Antes del ejercicio en vivo, creamos un mini-esquema **Capitanes / Botes / Reservas** (el clásico de los slides) para ilustrar qué muestra `EXPLAIN`.

In [ ]:
run("""
DROP TABLE IF EXISTS Reservas;
DROP TABLE IF EXISTS Capitanes;
DROP TABLE IF EXISTS Botes;

CREATE TABLE Capitanes (
    cid     SERIAL PRIMARY KEY,
    cnombre TEXT NOT NULL,
    rating  INT  NOT NULL,
    edad    INT
);

CREATE TABLE Botes (
    bid     SERIAL PRIMARY KEY,
    bnombre TEXT NOT NULL,
    bcolor  TEXT
);

CREATE TABLE Reservas (
    cid    INT NOT NULL REFERENCES Capitanes(cid),
    bid    INT NOT NULL REFERENCES Botes(bid),
    fecha  DATE NOT NULL
);

-- 1.000 capitanes, ratings 1-10
INSERT INTO Capitanes (cnombre, rating, edad)
SELECT 'Cap ' || i,
       1 + (random() * 9)::int,
       20 + (random() * 50)::int
FROM generate_series(1, 1000) AS i;

-- 50 botes en colores variados
INSERT INTO Botes (bnombre, bcolor)
SELECT 'Bote ' || i,
       (ARRAY['rojo','azul','verde','amarillo','blanco'])[(i % 5) + 1]
FROM generate_series(1, 50) AS i;

-- 3.000 reservas
INSERT INTO Reservas (cid, bid, fecha)
SELECT 1 + (random() * 999)::int,
       1 + (random() * 49)::int,
       DATE '2024-01-01' + (random() * 365)::int
FROM generate_series(1, 3000);

ANALYZE;
""")
print("Esquema Capitanes/Botes/Reservas listo.")

### EXPLAIN: solo el plan estimado

In [ ]:
explain("EXPLAIN SELECT * FROM Capitanes WHERE rating > 8;")

### EXPLAIN ANALYZE: ejecuta y mide tiempos reales

In [ ]:
explain("EXPLAIN ANALYZE SELECT * FROM Capitanes WHERE rating > 8;")

### EXPLAIN (ANALYZE, BUFFERS): páginas leídas de buffer vs. disco

In [ ]:
explain("EXPLAIN (ANALYZE, BUFFERS) SELECT * FROM Capitanes WHERE rating > 8;")

### Cuidado con escrituras: envolver en `BEGIN ... ROLLBACK`

`EXPLAIN ANALYZE` con `INSERT/UPDATE/DELETE` **sí ejecuta** la operación. La transacción permite ver el plan sin perder datos.

In [ ]:
# Para envolver en transacción, desactivamos autocommit por un momento.
conn.autocommit = False
try:
    explain("EXPLAIN ANALYZE DELETE FROM Reservas WHERE fecha < '2024-02-01';")
finally:
    conn.rollback()
    conn.autocommit = True
print("\n(rollback aplicado: las reservas siguen ahí)")

## Parte 2 · EXPLAIN de un join

In [ ]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT C.cnombre, R.fecha
FROM Capitanes C JOIN Reservas R ON C.cid = R.cid
WHERE C.rating > 8;
""")

**Preguntas para discutir en clase:**
1. ¿PostgreSQL eligió Hash Join, Nested Loop o Merge Join?
2. ¿La selección `rating > 8` se aplicó **antes** del join (push-down)?
3. ¿Cuál tabla quedó en el build side y cuál en el probe side? ¿Por qué?
4. ¿La estimación de filas se acercó a la realidad?

## Parte 3 · Ejercicio en vivo

Aquí hacemos el ejercicio del slide deck con tablas más grandes (1M empleados) para ver el efecto de los índices con datos en serio.

### Setup: empleados y departamentos

In [ ]:
run("""
DROP TABLE IF EXISTS empleados;
DROP TABLE IF EXISTS departamentos;

CREATE TABLE empleados (
  eid SERIAL PRIMARY KEY,
  nombre TEXT, depto_id INT, salario INT
);
CREATE TABLE departamentos (
  did SERIAL PRIMARY KEY,
  nombre TEXT, ciudad TEXT
);
""")
print("Tablas creadas.")

In [ ]:
run("""
INSERT INTO departamentos (nombre, ciudad)
SELECT 'Depto ' || i,
       CASE WHEN i % 3 = 0 THEN 'Santiago'
            WHEN i % 3 = 1 THEN 'Valparaíso'
            ELSE 'Concepción' END
FROM generate_series(1, 200) AS i;

INSERT INTO empleados (nombre, depto_id, salario)
SELECT 'Emp ' || i, (i % 200) + 1,
       30000 + (random() * 70000)::int
FROM generate_series(1, 1000000) AS i;

ANALYZE empleados;
ANALYZE departamentos;
""")
print("1.000.000 empleados + 200 departamentos cargados.")

Verifiquemos los tamaños:

In [ ]:
run("""
SELECT 'empleados'      AS tabla, COUNT(*) AS filas FROM empleados
UNION ALL
SELECT 'departamentos'  AS tabla, COUNT(*) AS filas FROM departamentos;
""")

### Consulta sin índices adicionales

Solo existen los índices implícitos en las PKs. ¿Qué hace PostgreSQL?

In [ ]:
explain("""
EXPLAIN ANALYZE
SELECT e.nombre, d.nombre AS depto, e.salario
FROM empleados e JOIN departamentos d ON e.depto_id = d.did
WHERE e.salario > 80000 AND d.ciudad = 'Santiago';
""")

**Preguntas para la clase:**
1. ¿Qué tipo de join eligió PostgreSQL?
2. ¿Dónde aplicó las selecciones?
3. ¿Usó algún índice?
4. ¿Cuántas filas estimó vs. cuántas produjo realmente?

### Agregar un índice en `salario`

In [ ]:
run("CREATE INDEX idx_emp_salario ON empleados (salario);")
print("Índice idx_emp_salario creado.")

In [ ]:
explain("""
EXPLAIN ANALYZE
SELECT e.nombre, d.nombre AS depto, e.salario
FROM empleados e JOIN departamentos d ON e.depto_id = d.did
WHERE e.salario > 80000 AND d.ciudad = 'Santiago';
""")

**Preguntas:**
1. ¿Cambió el plan? ¿Usa el nuevo índice? ¿de qué forma?
2. ¿Mejoró el tiempo?

In [ ]:
#### Para ver cuanto filtra la selección por salario

run("""
SELECT
  COUNT(*) FILTER (WHERE salario > 80000) AS sobre_80k,
  COUNT(*)                                AS total,
  ROUND(100.0 * COUNT(*) FILTER (WHERE salario > 80000) / COUNT(*), 1)
    AS pct_sobre_80k
FROM empleados;
""")

### Agregar un índice en `depto_id`

In [ ]:
run("CREATE INDEX idx_emp_depto ON empleados (depto_id);")
print("Índice idx_emp_depto creado.")

In [ ]:
explain("""
EXPLAIN ANALYZE
SELECT e.nombre, d.nombre AS depto, e.salario
FROM empleados e JOIN departamentos d ON e.depto_id = d.did
WHERE e.salario > 80000 AND d.ciudad = 'Santiago';
""")

**Pregunta:** ¿el optimizador ahora usa **INLJ** con el índice en `depto_id`? ¿O todavía prefiere Hash Join? ¿Por qué?

### Umbral de selectividad

El índice en `salario` probablemente **no se usó** con `salario > 80000` (~20% de filas). ¿Y con un filtro más selectivo?

In [ ]:
explain("EXPLAIN ANALYZE SELECT * FROM empleados WHERE salario > 60000;")

In [ ]:
explain("EXPLAIN ANALYZE SELECT * FROM empleados WHERE salario > 95000;")

**Lección:** el optimizador solo usa un índice cuando la selectividad lo justifica. El umbral típico es ~5–15% de las filas.

### Estadísticas desactualizadas

Insertamos muchos empleados de alto salario (sesgo) y vemos qué pasa **antes** y **después** de correr `ANALYZE`.

In [ ]:
run("""
INSERT INTO empleados (nombre, depto_id, salario)
SELECT 'Nuevo ' || i, (i % 200) + 1, 90000 + (random() * 10000)::int
FROM generate_series(1, 500000) AS i;

ALTER TABLE empleados SET (autovacuum_enabled = false);
""")
print("500.000 empleados de alto salario insertados; autovacuum desactivado.")

**Sin** correr `ANALYZE` — las estadísticas siguen viendo la distribución vieja:

In [ ]:
explain("EXPLAIN ANALYZE SELECT * FROM empleados WHERE salario > 95000;")

Responder: ¿las filas estimadas vs. reales **divergen mucho**?

Ahora actualizamos estadísticas:

In [ ]:
run("ANALYZE empleados;")
print("Estadísticas actualizadas.")

In [ ]:
explain("EXPLAIN ANALYZE SELECT * FROM empleados WHERE salario > 95000;")

**Pregunta:** ¿cambió el plan? ¿Mejoraron las estimaciones?

In [ ]:
run("ALTER TABLE empleados SET (autovacuum_enabled = true);")

### Para forzar un plan diferente (opcional)

PostgreSQL permite deshabilitar estrategias para comparar. **Solo para aprendizaje** — en producción, dejar que el optimizador decida.

In [ ]:
run("SET enable_hashjoin = off;")
explain("""
EXPLAIN ANALYZE
SELECT e.nombre, d.nombre AS depto, e.salario
FROM empleados e JOIN departamentos d ON e.depto_id = d.did
WHERE e.salario > 80000 AND d.ciudad = 'Santiago';
""")

In [ ]:
run("SET enable_hashjoin = on; SET enable_mergejoin = off;")
explain("""
EXPLAIN ANALYZE
SELECT e.nombre, d.nombre AS depto, e.salario
FROM empleados e JOIN departamentos d ON e.depto_id = d.did
WHERE e.salario > 80000 AND d.ciudad = 'Santiago';
""")

In [ ]:
run("RESET ALL;")
print("Configuración restaurada.")

## Parte 4 · Ejercicio integrador

Las tablas Capitanes / Botes / Reservas están cargadas desde el Bloque 1. Probemos las tres consultas representativas y discutamos qué índices ayudarían.

**Q1: Reservas de un capitán específico**

In [ ]:
explain("EXPLAIN ANALYZE SELECT * FROM Reservas WHERE cid = 42;")

**Q2: Capitanes con rating alto que reservaron botes rojos**

In [ ]:
explain("""
EXPLAIN ANALYZE
SELECT C.cnombre
FROM Capitanes C, Reservas R, Botes B
WHERE C.cid = R.cid AND R.bid = B.bid
  AND C.rating > 9 AND B.bcolor = 'rojo';
""")

**Q3: Número de reservas por bote, ordenado**

In [ ]:
explain("""
EXPLAIN ANALYZE
SELECT B.bnombre, COUNT(*)
FROM Botes B JOIN Reservas R ON B.bid = R.bid
GROUP BY B.bnombre
ORDER BY COUNT(*) DESC;
""")

**Tareas:**

1. Para cada consulta, proponer índices que podrían ayudar.
2. Justificar con las fórmulas de costo de las semanas 7–9.
3. Verificar con `EXPLAIN ANALYZE` antes y después de crearlos.

Espacio abajo para experimentar:

In [ ]:
# run("CREATE INDEX ... ;")
# explain("EXPLAIN ANALYZE ... ;")

## Herramientas de producción 

### `pg_stat_user_indexes` — ¿qué índices nunca se usan?

In [ ]:
run("""
SELECT relname AS tabla, indexrelname AS indice, idx_scan, idx_tup_read
FROM pg_stat_user_indexes
ORDER BY idx_scan ASC, indexrelname;
""")

Los índices con `idx_scan = 0` después de un workload representativo son candidatos a eliminar (ocupan espacio y ralentizan escrituras sin aportar lecturas).

### Tamaño de las tablas e índices

In [ ]:
run("""
SELECT relname,
       pg_size_pretty(pg_relation_size(oid)) AS tamano
FROM pg_class
WHERE relnamespace = 'public'::regnamespace
  AND relkind IN ('r','i')
ORDER BY pg_relation_size(oid) DESC;
""")